In [0]:

# Banking Transaction Data Engineering Platform

# Source location
SOURCE_PATH = "/Volumes/workspace/banking_bronze/raw_files/"

# Catalog and schemas
CATALOG = "workspace"
BRONZE_SCHEMA = "banking_bronze"

# Bronze table names
CUSTOMERS_TABLE = "bronze_customers"
ACCOUNTS_TABLE = "bronze_accounts"

MOBILE_RAW_TABLE = "bronze_mobile_raw_json"
MOBILE_TRANSACTIONS_TABLE = "bronze_mobile_transactions"
MOBILE_EVENTS_TABLE = "bronze_mobile_events"

ATM_TRANSACTIONS_TABLE = "bronze_atm_transactions"
ATM_EVENTS_TABLE = "bronze_atm_events"

BRANCH_TRANSACTIONS_TABLE = "bronze_branch_transactions"
BRANCH_EVENTS_TABLE = "bronze_branch_events"

print("Banking project configuration loaded successfully.")
print("Source path:", SOURCE_PATH)


In [0]:
files = dbutils.fs.ls(SOURCE_PATH)

print("Files available in the source location: ")

for file in files:
    print(file.name)

In [0]:
from pyspark.sql import functions as F
customers_df = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(SOURCE_PATH + "customers.csv")
)

customers_bronze_df = (
    customers_df.withColumn("source_file", F.lit("customers.csv"))
    .withColumn("_ingestion_timestamp",F.current_timestamp())
)

customers_bronze_df.write\
                  .format("delta")\
                  .mode("overwrite")\
                  .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{CUSTOMERS_TABLE}")

print("Customers Bronze ingestion completed.")
print("Records: ",customers_bronze_df.count())

In [0]:
display(spark.table("workspace.banking_bronze.bronze_customers"))

In [0]:
accounts_df = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(SOURCE_PATH + "accounts.csv")
)

bronze_accounts_df = (
    accounts_df.withColumn("source_file", F.lit("accounts.csv"))
    .withColumn("_ingestion_time",F.current_timestamp())
)

bronze_accounts_df.write\
                  .format("delta")\
                  .mode("overwrite")\
                  .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{ACCOUNTS_TABLE}")

print("Accounts Bronze ingestion completed.")
print("Records: ",bronze_accounts_df.count())




In [0]:
display(spark.table("workspace.banking_bronze.bronze_accounts"))

In [0]:

mobile_json_path = SOURCE_PATH + "mobile_banking_transactions.json"

mobile_raw_df = spark.read.text(mobile_json_path)

display(mobile_raw_df)


In [0]:
mobile_raw_bronze_df = (
    mobile_raw_df.withColumn("source_file", F.lit("mobile_banking_transactions.json"))
    .withColumn("_ingestion_time",F.current_timestamp())
)

mobile_raw_bronze_df.write\
                  .format("delta")\
                  .mode("overwrite")\
                  .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{MOBILE_RAW_TABLE}")

print("Mobile banking raw  JSON ingestion completed.")
print("Records: ",mobile_raw_bronze_df.count())


In [0]:
mobile_json_text = "\n".join(
    row.value for row in mobile_raw_df.collect()
        )
print(mobile_json_text[:3000])

In [0]:
import json

mobile_data = json.loads(mobile_json_text)

mobile_transactions = mobile_data["transactions"]
mobile_events = mobile_data["transaction_events"]

print("Mobile transactions:", len(mobile_transactions))
print("Mobile transaction events:", len(mobile_events))

print("First Transaction:")
print(mobile_transactions[0])


print("First event:")
print(mobile_events[0])


In [0]:
from pyspark.sql.types import (StructField, StructType, StringType, IntegerType)
from pyspark.sql import functions as F
# Defining Mobile banking transaction schema
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("transaction_timestamp", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("transaction_reference", StringType(), True),
    StructField("related_transaction_id", StringType(), True)
])


# Defining Mobile Banking event schema

event_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("event_id", StringType(), True),
    StructField("event_sequence", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("source_system", StringType(), True)
])


# 3. Convert Python lists into Spark DataFrames

mobile_transactions_df = spark.createDataFrame(mobile_transactions,transaction_schema)

mobile_events_df = spark.createDataFrame(mobile_events,event_schema)

# Add  ingestion metadata

mobile_transactions_df = (
    mobile_transactions_df.withColumn("source_file", F.lit("mobile_banking_transactions.json"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
) 



mobile_events_df = (
    mobile_events_df.withColumn("source_file", F.lit("mobile_banking_transactions.json"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
) 

# Saving both dataFrames as Bronze delta Tables

mobile_transactions_df.write.format("delta")\
                            .mode("overwrite")\
                            .saveAsTable("workspace.banking_bronze.bronze_mobile_transactions")

mobile_events_df.write.format("delta")\
                            .mode("overwrite")\
                            .saveAsTable("workspace.banking_bronze.bronze_mobile_events")


print("Mobile Banking Bronze tables created successfully.")
print("Transaction records:", mobile_transactions_df.count())
print("Event records:", mobile_events_df.count())




In [0]:
# Read the saved Bronze tables

mobile_transactions_check = spark.table(
    "workspace.banking_bronze.bronze_mobile_transactions"
)

mobile_events_check = spark.table(
    "workspace.banking_bronze.bronze_mobile_events"
)


# Verify transaction data
print("Mobile Transactions:")
print("Count:", mobile_transactions_check.count())
mobile_transactions_check.printSchema()
display(mobile_transactions_check)

# Verify event data
print("Mobile Transactions Events:")
print("Count:",mobile_events_check.count())
mobile_events_check.printSchema()
display(mobile_events_check)


In [0]:
# Defining the ATM transactions file path 

atm_transactions_path = SOURCE_PATH + "atm_banking_transactions.csv"

# Read the CSV while preserving source values as strings

atm_transactions_df = (
    spark.read
    .option("header","true")
    .option("inferSchema","False")
    .csv(atm_transactions_path)
)

# Add ingestion metadata
atm_transactions_bronze_df = (
    atm_transactions_df
    .withColumn("_source_file", F.lit("atm_banking_transactions.csv"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as a Bronze Delta table

atm_transactions_bronze_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.banking_bronze.bronze_atm_transactions")


print("ATM transactions ingested successfully.")
print("Records:", atm_transactions_bronze_df.count())

In [0]:
files = dbutils.fs.ls(SOURCE_PATH)
for file in files:
    print(file.name)

In [0]:
atm_transactions_check = spark.table("workspace.banking_bronze.bronze_atm_transactions")

print("ATM record counts:", atm_transactions_check.count())
atm_transactions_check.printSchema()
display(atm_transactions_check)

In [0]:
# Defining ATM events file path
from pyspark.sql import functions as F
atm_events_path  = SOURCE_PATH + "atm_banking_events.csv"

# Read the csv , preserving source values as String

atm_events_df = (
    spark.read
    .option("header","true")
    .option("inferSchema","false")
    .csv(atm_events_path)
)

# Add ingestion metadata
atm_events_bronze_df = (
    atm_events_df.withColumn("source_file", F.lit("atm_banking_events.csv"))
                 .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as a Bronze delta Table
atm_events_bronze_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.banking_bronze.bronze_atm_events")

print("ATM events ingested successfully.")
print("Records:", atm_events_bronze_df.count())

In [0]:
atm_events_check = spark.table("workspace.banking_bronze.bronze_atm_events")

print("ATM record count: ",atm_events_check.count())
atm_events_check.printSchema()
display(atm_events_check)

In [0]:
# Define the branch transactions file path
branch_transactions_path = SOURCE_PATH + "branch_banking_transactions.csv"

# Read the csv , preserving source value as strings
branch_transacations_df = (
    spark.read
    .option("header","true")
    .option("inferSchema","false")
    .csv(branch_transactions_path)
)

# Add ingestion metadata
branch_transactions_bronze_df = (
    branch_transacations_df
    .withColumn("_source_file", F.lit("branch_banking_transactions.csv"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as a Bronze delta Table
branch_transactions_bronze_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.banking_bronze.bronze_branch_transactions")

print("Brance Transactions ingested successfully.")
print("Records:", branch_transactions_bronze_df.count())

In [0]:
branch_transactions_check = spark.table("workspace.banking_bronze.bronze_branch_transactions")

print("Branch transaction records:", branch_transactions_check.count())

branch_transactions_check.printSchema()

display(branch_transactions_check)


In [0]:
# Define the branch events file path
branch_events_path = SOURCE_PATH + "branch_banking_events.csv"

# Read the csv , preserving source value as strings
branch_events_df = (
    spark.read
    .option("header","true")
    .option("inferSchema","false")
    .csv(branch_events_path)
)

# Add ingestion metadata
branch_events_bronze_df = (
    branch_events_df
    .withColumn("_source_file", F.lit("branch_banking_events.csv"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as a Bronze delta Table
branch_events_bronze_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.banking_bronze.bronze_branch_events")

print("Brance events ingested successfully.")
print("Records:", branch_events_bronze_df.count())

In [0]:
branch_events_check = spark.table("workspace.banking_bronze.bronze_branch_events")

print("Branch events records:", branch_events_check.count())

branch_events_check.printSchema()

display(branch_events_check)

In [0]:

# Expected record counts from our ingestion steps
expected_counts = {
    "bronze_customers": 8,
    "bronze_accounts": 9,
    "bronze_mobile_raw_json": 207,
    "bronze_mobile_transactions": 10,
    "bronze_mobile_events": 10,
    "bronze_atm_transactions": 10,
    "bronze_atm_events": 15,
    "bronze_branch_transactions": 10,
    "bronze_branch_events": 12
}

# Validate all Bronze tables
print("BRONZE LAYER VALIDATION")
print("-" * 65)

total_records = 0
all_passed = True

for table_name, expected in expected_counts.items():

    full_table_name = f"workspace.banking_bronze.{table_name}"

    actual = spark.table(full_table_name).count()

    status = "PASS" if actual == expected else "FAIL"

    if status == "FAIL":
        all_passed = False

    print(
        f"{table_name:<35} "
        f"Expected: {expected:<5} "
        f"Actual: {actual:<5} "
        f"{status}"
    )

    total_records += actual

print("-" * 65)
print("Total records/lines:", total_records)
print("Overall validation:", "PASSED" if all_passed else "FAILED")
